# Monte Carlo Probabilistic Scheduling

This notebook demonstrates `trueppm-scheduler`'s Monte Carlo simulation:
three-point PERT estimates, running 1 000 simulations (OSS tier limit), and interpreting
P50/P80/P95 completion dates.

**Install**
```bash
pip install trueppm-scheduler matplotlib
```

In [ ]:
from datetime import date, timedelta
from collections import Counter

from trueppm_scheduler import (
    Calendar, Dependency, Project, Task, monte_carlo, schedule,
)

## 1. Add three-point PERT estimates to tasks

Set `optimistic_duration`, `most_likely_duration`, and `pessimistic_duration`
on any task where you want stochastic sampling. Tasks without these fields
use their deterministic `duration` every run.

In [ ]:
def days(n: int) -> timedelta:
    return timedelta(days=n)

tasks = [
    Task(
        id="design", name="Design",
        duration=days(5),
        optimistic_duration=days(3),
        most_likely_duration=days(5),
        pessimistic_duration=days(10),
    ),
    Task(
        id="build", name="Build",
        duration=days(15),
        optimistic_duration=days(10),
        most_likely_duration=days(15),
        pessimistic_duration=days(25),
    ),
    Task(
        id="test", name="Test",
        duration=days(8),
        optimistic_duration=days(5),
        most_likely_duration=days(8),
        pessimistic_duration=days(15),
    ),
    # No PERT estimates — uses deterministic duration every run
    Task(id="deploy", name="Deploy", duration=days(2)),
]

dependencies = [
    Dependency(predecessor_id="design", successor_id="build"),
    Dependency(predecessor_id="design", successor_id="test"),
    Dependency(predecessor_id="build",  successor_id="deploy"),
    Dependency(predecessor_id="test",   successor_id="deploy"),
]

project = Project(
    id="release-mc",
    name="Release v1.0 (Monte Carlo)",
    start_date=date(2026, 4, 1),
    tasks=tasks,
    dependencies=dependencies,
    calendar=Calendar(),
)

## 2. Run the deterministic CPM baseline first

The CPM result uses `duration` (most likely) — this is the single-point estimate
that traditional Gantt charts show.

In [ ]:
cpm_result = schedule(project)
print(f"CPM (deterministic) finish: {cpm_result.project_finish}")
print(f"Critical path: {' → '.join(cpm_result.critical_path)}")

## 3. Run Monte Carlo simulation

In [ ]:
mc_result = monte_carlo(project, runs=1_000, seed=42)

print(f"Runs:  {mc_result.runs:,}")
print(f"P50:   {mc_result.p50}  (50% of runs finish by this date)")
print(f"P80:   {mc_result.p80}  (80% of runs finish by this date — recommended commitment date)")
print(f"P95:   {mc_result.p95}  (95% of runs finish by this date — contractual deadline buffer)")
print()
cpm_finish = cpm_result.project_finish
p80_slip = (mc_result.p80 - cpm_finish).days
print(f"CPM finish: {cpm_finish}")
print(f"P80 vs CPM: +{p80_slip} calendar days ({p80_slip/7:.1f} weeks of schedule risk)")

## 4. Visualise the completion date distribution

The full sorted distribution is in `mc_result.distribution`.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    from datetime import datetime

    # Count completions per week (bucket by Monday of each week)
    def week_start(d: date) -> date:
        return d - timedelta(days=d.weekday())

    counts: Counter[date] = Counter(week_start(d) for d in mc_result.distribution)
    weeks = sorted(counts)
    freqs = [counts[w] / mc_result.runs * 100 for w in weeks]  # % of runs

    fig, ax = plt.subplots(figsize=(10, 4))
    bar_dates = [datetime(w.year, w.month, w.day) for w in weeks]
    ax.bar(bar_dates, freqs, width=5, color="#6B7280", alpha=0.7, label="Simulations (%)")

    # Percentile lines
    for label, d, color, ls in [
        ("P50", mc_result.p50, "#16a34a", "-"),
        ("P80", mc_result.p80, "#d97706", "--"),
        ("P95", mc_result.p95, "#dc2626", ":"),
    ]:
        ax.axvline(datetime(d.year, d.month, d.day), color=color,
                   linestyle=ls, linewidth=1.5, label=f"{label}: {d}")

    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %d"))
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=0, interval=2))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha="right")
    ax.set_xlabel("Week of completion")
    ax.set_ylabel("% of simulations")
    ax.set_title("Monte Carlo completion date distribution")
    ax.legend()
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — pip install matplotlib to see the histogram")
    print("Completion range:", mc_result.distribution[0], "→", mc_result.distribution[-1])

## 5. Interpreting the results

| Percentile | Meaning | Recommended use |
|-----------|---------|----------------|
| **P50** | 50% chance of finishing by this date | Internal target / team commitment |
| **P80** | 80% chance of finishing by this date | Stakeholder commitment / sprint goal |
| **P95** | 95% chance of finishing by this date | Contractual deadline / SLA |

The CPM deterministic finish is typically close to P50 — meaning there is only
a **50% chance** the project finishes on the date shown in a traditional Gantt chart.
Committing to the P80 date adds a buffer that reflects realistic schedule risk.

## 6. Reproducibility and sensitivity

Pass `seed=` for reproducible results. Vary the pessimistic estimate to model
different risk scenarios.

In [ ]:
# Optimistic scenario: build pessimistic = 20 days (vs 25)
tasks_opt = [
    Task(
        id=t.id, name=t.name, duration=t.duration,
        optimistic_duration=t.optimistic_duration,
        most_likely_duration=t.most_likely_duration,
        pessimistic_duration=(
            timedelta(days=20) if t.id == "build" else t.pessimistic_duration
        ),
    )
    for t in tasks
]

project_opt = Project(
    id="release-mc-opt", name="Optimistic scenario",
    start_date=date(2026, 4, 1),
    tasks=tasks_opt, dependencies=dependencies, calendar=Calendar(),
)

mc_opt = monte_carlo(project_opt, runs=1_000, seed=42)
print("Base scenario:      P80 =", mc_result.p80)
print("Optimistic scenario: P80 =", mc_opt.p80)

## 7. CLI usage

```bash
# Run Monte Carlo from a JSON project file
trueppm-scheduler monte-carlo --input project.json

# JSON output + full distribution
trueppm-scheduler monte-carlo --input project.json --json --distribution
```

The `--distribution` flag includes the full sorted date list in the JSON output,
suitable for rendering a histogram in a frontend application.